In [6]:
#MultiAsset and MultiTimeframe BackTesting Framework

import ccxt
import pandas as pd
import matplotlib.pyplot as plt

SYMBOLS = ['BTC/USDT', 'ETH/USDT', 'SOL/USDT', 'BNB/USDT']

TIMEFRAMES = ['5m', '15m', '1h']

def download_data(symbol , timeframe , limit=5000):
    exchange = ccxt.binance()
    ohlcv = exchange.fetch_ohlcv(symbol = symbol, timeframe = timeframe, limit = limit)
    df = pd.DataFrame(ohlcv, columns = ['Time', 'Open', 'High', 'Low', 'Close', 'Volume'])
    df['Time'] = pd.to_datetime(df['Time'], unit='ms')
    return df

# --------------------------------
# 3. Session VWAP
# --------------------------------
def calculate_vwap(df):

    typical_price = (
        df['High']
        + df['Low']
        + df['Close']
    ) / 3

    tpv = typical_price * df['Volume']

    # Reset each UTC day
    date_group = df.index.date

    cumulative_tpv = tpv.groupby(date_group).cumsum()

    cumulative_volume = (df['Volume'].groupby(date_group).cumsum())

    df['VWAP'] = (cumulative_tpv / cumulative_volume)

    return df

# --------------------------------
# 4. ZScore
# --------------------------------
def calculate_zscore(df):

    deviation = (
        (df['Close'] - df['VWAP'])
        / df['VWAP']
    ) * 100

    rolling_mean = (deviation.rolling(50).mean())

    rolling_std = (deviation.rolling(50).std())

    df['ZScore'] = (deviation - rolling_mean) / rolling_std

    return df

results = []

for symbol in SYMBOLS:

    for timeframe in TIMEFRAMES:
        print(
            f"Testing "
            f"{symbol} "
            f"{timeframe}"
        )
        df = download_data(symbol=symbol, timeframe=timeframe)
        df.set_index('Time', inplace=True)
        calculate_vwap(df)
        calculate_zscore(df)
        # --------------------------------
        # SIGNALS
        # --------------------------------
        BUY_ZSCORE = -1.8
        SELL_ZSCORE = -0.5

        df['Signal'] = 0

        df.loc[(df['ZScore'] < BUY_ZSCORE) , 'Signal'] = 1
        df.loc[df['ZScore'] > SELL_ZSCORE,'Signal'] = -1

        # --------------------
        # BACKTEST
        # --------------------
        INITIAL_CAPITAL = 5000
        TRADING_FEE = 0.0008

        balance = INITIAL_CAPITAL
        position = 0
        entry_price = 0

        total_trades = 0
        winning_trades = 0
        
        for i in range(len(df)):
            row = df.iloc[i]
            signal = row['Signal']
            close_price = row['Close']
            if signal == 1 and position == 0:
                position = 1
                entry_price = close_price

            elif signal == -1 and position == 1:
                pnl_pct = (close_price - entry_price) / entry_price
                gross_pnl = (balance * pnl_pct)

                fee_cost = (balance * TRADING_FEE)

                pnl = (gross_pnl - fee_cost)

                balance += pnl

                total_trades += 1

                if pnl > 0:
                    winning_trades += 1

                position = 0
                
        win_rate = (winning_trades / total_trades * 100 if total_trades > 0 else 0)
                
        results.append({

                'Asset': symbol,

                'Timeframe': timeframe,

                'Final Balance': round(balance, 2),

                'Return %': round((( balance / INITIAL_CAPITAL ) - 1) * 100, 2),

                'Trades': total_trades,

                'Win Rate': round(win_rate, 2)})


results_df = pd.DataFrame(
    results
)

results_df.sort_values(

    by='Return %',

    ascending=False

)

Testing BTC/USDT 5m
Testing BTC/USDT 15m
Testing BTC/USDT 1h
Testing ETH/USDT 5m
Testing ETH/USDT 15m
Testing ETH/USDT 1h
Testing SOL/USDT 5m
Testing SOL/USDT 15m
Testing SOL/USDT 1h
Testing BNB/USDT 5m
Testing BNB/USDT 15m
Testing BNB/USDT 1h


,Asset,Timeframe,Final Balance,Return %,Trades,Win Rate
8,SOL/USDT,1h,5282.05,5.64,14,64.29
2,BTC/USDT,1h,5199.52,3.99,14,64.29
5,ETH/USDT,1h,5185.98,3.72,16,56.25
6,SOL/USDT,5m,5119.29,2.39,15,73.33
10,BNB/USDT,15m,5084.63,1.69,14,57.14
3,ETH/USDT,5m,5056.04,1.12,16,75.00
9,BNB/USDT,5m,5042.16,0.84,12,58.33
0,BTC/USDT,5m,5040.77,0.82,16,68.75
11,BNB/USDT,1h,5017.23,0.34,13,69.23
7,SOL/USDT,15m,4912.19,-1.76,13,61.54
